# REPORT inline data (thesis text)

Aggregate statistics complementing `REPORT_plots_FINAL.ipynb`. Each section mirrors the plots notebook; outputs are printed text and DataFrames for use in the main text.

- Data: same FINAL checkpoints + many-seed MNIST as the plots notebook
- Computations: `analysis/final/report_plots/inline_stats.py`

In [25]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def _find_project_root(start: Path | None = None) -> Path:
	p = (start or Path.cwd()).resolve()
	for cand in [p, *p.parents]:
		if (cand / "pyproject.toml").is_file():
			return cand
	return p


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
	sys.path.insert(0, str(PROJECT_ROOT))

from analysis.final.final_analysis_config import (
	DEFAULT_SCORE_FACTORS,
	EXPERT_MODEL_DIR_BY_DATASET,
	save_dir_for_dataset,
)
from analysis.final.final_experiment_display import ExperimentDisplayLabels
from analysis.final.load_final_analysis import run_final_analysis
from analysis.final.report_plots.config import ReportSummarySemAxis
from analysis.final.report_plots.inline_stats import (
	print_novelty_bss_stats,
	print_novelty_non_control_dataset_stats,
	print_novelty_period_count_stats,
	print_optimizer_profile_theta_stats,
)
from analysis.final.report_plots.report_plots_helper import (
	build_report_period_count_runs_mixed,
)

_EXPER_LABEL_SHUFFLE_ALL = r"$\text{Shuffle}_{all}$"
_EXPER_LABEL_SHUFFLE_INTERLEAVED = r"$\text{Shuffle}_{interleaved}$"
_EXPER_LABEL_SHUFFLE_SEQ = r"$\text{Shuffle}_{seq}$"
LABELS = ExperimentDisplayLabels(
	shuffle_all=_EXPER_LABEL_SHUFFLE_ALL,
	shuffle_interleaved=_EXPER_LABEL_SHUFFLE_INTERLEAVED,
	shuffle_nopre="Control global (✗PT)",
	seq_nopre="Control sequential (✗PT)",
	seq_pre="Control sequential (✔PT)",
	shuffle_seq=_EXPER_LABEL_SHUFFLE_SEQ,
	no_pretrain=" Control (no pretrain)",
)

N_CHECKPOINT_SAMPLES = 20
SCORE_FACTORS = dict(DEFAULT_SCORE_FACTORS)
SCORE_FACTORS["n_partial_decay_score"] = 0.0
SCORE_FACTORS["n_active_decay_score"] = 0.0
RESCORE = False
MINMAX_NORMALIZE_ROBUSTNESS = True
REPARSE_DIGIT_A = False
STRICT = False
BTSP_START_STRATEGY = "acceleration"
USE_REAL_PROGRESSION = True

# Match REPORT_plots_FINAL.ipynb pooling for summary bars.
REPORT_SUMMARY_SEM_AXIS: ReportSummarySemAxis = "all"

In [27]:
# FINAL checkpoints: MNIST supplements many_seeds gaps; CIFAR is single-seed throughout.
all_results: dict = {}
errors_all: list[dict] = []

for DATASET in ("mnist", "cifar10"):
	SAVE_DIR = str(save_dir_for_dataset(PROJECT_ROOT, DATASET))
	Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
	ar, tree, errors, _ds_key, model_ids = run_final_analysis(
		input_dir=PROJECT_ROOT / "results",
		output_dir=Path(SAVE_DIR),
		project_root=PROJECT_ROOT,
		expert_model_dir=EXPERT_MODEL_DIR_BY_DATASET[DATASET],
		n_checkpoint_samples=N_CHECKPOINT_SAMPLES,
		score_factors=SCORE_FACTORS,
		rescore=RESCORE,
		minmax_normalize_robustness=MINMAX_NORMALIZE_ROBUSTNESS,
		reparse_digit_a=REPARSE_DIGIT_A,
		strict=STRICT,
		btsp_start_strategy=BTSP_START_STRATEGY,
		use_real_progression=USE_REAL_PROGRESSION,
		dataset=DATASET,
		verbose_errors=True,
	)
	all_results.update(ar)
	errors_all.extend(errors or [])
	print(f"DATASET={DATASET} model_ids={model_ids} experiments={len(tree)} runs={len(ar)}")

if errors_all:
	display(pd.DataFrame(errors_all))

df_period_runs = build_report_period_count_runs_mixed(PROJECT_ROOT, all_results)
n_mnist = (df_period_runs["model_id"] == "dnn_5x64").sum()
n_cifar = len(df_period_runs) - n_mnist
print(
	f"summary run rows: total={len(df_period_runs)}  "
	f"mnist={n_mnist}  cifar={n_cifar}"
)

Experiments:   0%|          | 0/10 [00:00<?, ?it/s]

Robustness already min-max normalized in cached runs; skipped min-max normalization
DATASET=mnist model_ids=['dnn_5x64'] experiments=10 runs=50


Experiments:   0%|          | 0/16 [00:00<?, ?it/s]

Robustness already min-max normalized in cached runs; skipped min-max normalization
DATASET=cifar10 model_ids=['inception_small_dnn_5x64', 'resnet32_dnn_5x64'] experiments=16 runs=100
summary run rows: total=420  mnist=320  cifar=100


## BTSP requires novelty

Run-level period counts pooled over MNIST (multi-seed) and CIFAR (single seed), matching the six-group summary figure in `REPORT_plots_FINAL.ipynb`.

For each category (**Global**, **Sequential**): mean period count for control (✗PT), control (✔PT), and non-control. Welch t-test: control (✔PT) vs non-control. Inset: mean BTSP period length (iterations) for Control global (✔PT), pooled over all periods in that group.

BSS (`total_score`, one sample per BTSP period from FINAL checkpoints):

1. **Global control vs non-control:** Control global (✗PT) vs non-control global (+ Welch t-test)
2. **Not control (✔PT) by category:** all Global experiments except Control global (✔PT) vs all Sequential experiments except Control sequential (✔PT) (+ Welch t-test)

Non-control breakdown (period counts; flat mean over MNIST optimizers × seeds; CIFAR flat over both architectures):

- **Global:** $\text{Shuffle}_{\{0,1\}}$; pool($\text{Shuffle}_{interleaved}$, $\text{Shuffle}_{all}$)
- **Sequential:** $\text{Shuffle}_{seq}$; pool(Recover, Reinforce)

Welch t-test MNIST vs CIFAR for each row. Architecture table: pairwise tests across FCN / Inception / ResNet within each experiment (Holm per experiment). Experiment table: all pairwise comparisons between the three individual experiments in each category (Global / Sequential), per architecture (Holm per category, 3 tests each).

In [28]:
novelty_summary, novelty_ttest, novelty_inset = print_novelty_period_count_stats(
	df_period_runs,
	sem_axis=REPORT_SUMMARY_SEM_AXIS,
	project_root=PROJECT_ROOT,
	all_results=all_results,
)

print("\n--- summary table ---")
display(
	novelty_summary[
		[
			"category",
			"label",
			"n_runs",
			"mean",
			"sem",
			"ci_lo",
			"ci_hi",
		]
	]
)

print("\n--- t-test table ---")
display(
	novelty_ttest[
		[
			"category",
			"control_yes_pt_mean",
			"non_control_mean",
			"mean_diff",
			"t_stat",
			"p_value",
			"significant_005",
			"n_control_yes_pt",
			"n_non_control",
		]
	]
)

if novelty_inset is not None:
	print("\n--- inset period length ---")
	display(
		novelty_inset[
			[
				"label",
				"n_periods",
				"mean",
				"sem",
				"ci_lo",
				"ci_hi",
			]
		]
	)

print("\n" + "=" * 72)
novelty_bss_summary, novelty_bss_ttest = print_novelty_bss_stats(all_results)

print("\n--- BSS summary table ---")
display(
	novelty_bss_summary[
		[
			"section",
			"label",
			"n_periods",
			"mean",
			"sem",
			"ci_lo",
			"ci_hi",
		]
	]
)

print("\n--- BSS t-test table ---")
display(
	novelty_bss_ttest[
		[
			"comparison",
			"mean_a",
			"mean_b",
			"mean_diff",
			"t_stat",
			"p_value",
			"significant_005",
			"n_a",
			"n_b",
		]
	]
)

print("\n" + "=" * 72)
(
	novelty_nc_summary,
	novelty_nc_category_means,
	novelty_nc_ttest,
	novelty_nc_pairwise,
	novelty_nc_wide,
	novelty_nc_latex,
	novelty_nc_experiment_wide,
	novelty_nc_experiment_latex,
) = print_novelty_non_control_dataset_stats(
	df_period_runs,
	LABELS,
)

print("\n--- non-control category pooled MNIST/CIFAR means ---")
display(novelty_nc_category_means)

print("\n--- non-control MNIST/CIFAR summary ---")
display(
	novelty_nc_summary[
		[
			"category",
			"label",
			"dataset",
			"n_runs",
			"mean",
			"sem",
			"ci_lo",
			"ci_hi",
		]
	]
)

print("\n--- non-control MNIST/CIFAR t-test ---")
display(
	novelty_nc_ttest[
		[
			"category",
			"label",
			"n_mnist",
			"n_cifar",
			"mnist_mean",
			"cifar_mean",
			"mean_diff",
			"t_stat",
			"p_value",
			"significant_005",
		]
	]
)

print("\n--- non-control architecture pairwise t-test ---")
display(
	novelty_nc_pairwise[
		[
			"category",
			"label",
			"comparison",
			"n_a",
			"n_b",
			"mean_a",
			"mean_b",
			"mean_diff",
			"t_stat",
			"p_value",
			"p_value_holm",
			"significant_005",
		]
	]
)

print("\n--- non-control architecture wide table ---")
display(novelty_nc_wide)

print("\n--- non-control architecture LaTeX ---")
print(novelty_nc_latex)

print("\n--- non-control experiment-pair wide table ---")
display(novelty_nc_experiment_wide)

print("\n--- non-control experiment-pair LaTeX ---")
print(novelty_nc_experiment_latex)

Global:
  Control (✗PT): mean=200.067 (±1.96·SEM=55.736, n=45)
  Control (✔PT): mean=5.333 (±1.96·SEM=2.252, n=15)
  Non-control: mean=12.378 (±1.96·SEM=1.152, n=135)
Sequential:
  Control (✗PT): mean=116.667 (±1.96·SEM=10.346, n=45)
  Control (✔PT): mean=22.067 (±1.96·SEM=2.761, n=45)
  Non-control: mean=82.948 (±1.96·SEM=6.604, n=135)

Control (✔PT) vs non-control (Welch t-test):
  Global: Δmean=-7.044 (control=5.333, non-control=12.378), t=-2.784, p=1.08e-02, p<0.05=yes (n=15/135)
  Sequential: Δmean=-60.881 (control=22.067, non-control=82.948), t=-8.505, p=9.24e-15, p<0.05=yes (n=45/135)

Inset period length (Control global (✔PT)): mean=34.100 (±1.96·SEM=5.684, n_periods=80)

--- summary table ---


,category,label,n_runs,mean,sem,ci_lo,ci_hi
0,Global,Control (✗PT),45,200.066667,55.736441,90.823241,309.310092
1,Global,Control (✔PT),15,5.333333,2.252336,0.918755,9.747911
2,Global,Non-control,135,12.377778,1.152346,10.119180,14.636376
3,Sequential,Control (✗PT),45,116.666667,10.346424,96.387676,136.945657
4,Sequential,Control (✔PT),45,22.066667,2.761276,16.654565,27.478768
5,Sequential,Non-control,135,82.948148,6.604330,70.003662,95.892635



--- t-test table ---


,category,control_yes_pt_mean,non_control_mean,mean_diff,t_stat,p_value,significant_005,n_control_yes_pt,n_non_control
0,Global,5.333333,12.377778,-7.044444,-2.784362,1.077570e-02,True,15,135
1,Sequential,22.066667,82.948148,-60.881481,-8.504973,9.243228e-15,True,45,135



--- inset period length ---


...


--- non-control architecture wide table ---


,category,label,mean_fcn,mean_inception,mean_resnet,p_fcn_inception,p_fcn_resnet,p_inception_resnet
0,Global,"$\text{Shuffle}_{\{0,1\}}$",4.571429,3.2,4.0,6.520214e-01,1.000000e+00,1.000000
1,Global,"Pool ($\text{Shuffle}_{interleaved}$, $\text{S...",20.000000,5.5,2.0,1.503995e-06,7.356755e-15,0.068652
2,Sequential,$\text{Shuffle}_{seq}$,43.742857,28.0,18.6,2.909803e-02,2.971024e-04,0.060554
3,Sequential,"Pool (Recover, Reinforce)",124.400000,42.8,29.8,2.753014e-09,1.520643e-10,0.142038



--- non-control architecture LaTeX ---
\begin{table}[ht]
\centering
\small
\resizebox{0.5\linewidth}{!}{%
\begin{tabular}{lrrrrrr}
\toprule
Experiment & $\mu_{(\mathrm{FCN})}$ & $\mu_{(\mathrm{Inc.})}$ & $\mu_{(\mathrm{ResNet})}$ & $p^{(\mathrm{FCN})}_{(\mathrm{Inc.})}$ & $p^{(\mathrm{FCN})}_{(\mathrm{Res.})}$ & $p^{(\mathrm{Inc.})}_{(\mathrm{Res.})}$ \\
\midrule
$\text{Shuffle}_{\{0,1\}}$ & 4.6 & 3.2 & 4.0 & 0.652 & 1.000 & 1.000 \\
\{$\text{Shuffle}_{interleaved}$, $\text{Shuffle}_{all}$} & 20.0 & 5.5 & 2.0 & $^{{*}{*}{*}}$ & $^{{*}{*}{*}}$ & 0.069 \\
$\text{Shuffle}_{seq}$ & 43.7 & 28.0 & 18.6 & $^{{*}}$ & $^{{*}{*}{*}}$ & 0.061 \\
\{Recover, Reinforce} & 124.4 & 42.8 & 29.8 & $^{{*}{*}{*}}$ & $^{{*}{*}{*}}$ & 0.142 \\
\bottomrule
\end{tabular}%
}
\caption{Non-control period counts by architecture. Pairwise comparisons use Holm-corrected $p$-values within each experiment; cells show $^{{*}}p<0.05$, $^{{*}{*}}p<0.01$, $^{{*}{*}{*}}p<0.001$, or the Holm $p$-value otherwise.}
\label{t

,category,label_a,label_b,p_fcn,p_inception,p_resnet
0,Global,"$\text{Shuffle}_{\{0,1\}}$",$\text{Shuffle}_{interleaved}$,1.413133e-05,0.192739,0.151107
1,Global,"$\text{Shuffle}_{\{0,1\}}$",$\text{Shuffle}_{all}$,2.403058e-08,0.034963,0.514843
2,Global,$\text{Shuffle}_{interleaved}$,$\text{Shuffle}_{all}$,4.365466e-03,0.017664,0.134448
3,Sequential,$\text{Shuffle}_{seq}$,Recover,1.745957e-05,0.466565,0.561924
4,Sequential,$\text{Shuffle}_{seq}$,Reinforce,6.366509e-06,0.466565,0.954131
5,Sequential,Recover,Reinforce,5.528212e-01,0.948804,0.954131



--- non-control experiment-pair LaTeX ---
\begin{table}[ht]
\centering
\small
\resizebox{0.5\linewidth}{!}{%
\begin{tabular}{lllccc}
\toprule
Category & Exp. A & Exp. B & $p^{(\mathrm{FCN})}$ & $p^{(\mathrm{Inc.})}$ & $p^{(\mathrm{Res.})}$ \\
\midrule
Global & $\text{Shuffle}_{\{0,1\}}$ & $\text{Shuffle}_{interleaved}$ & $^{{*}{*}{*}}$ & 0.193 & 0.151 \\
 & $\text{Shuffle}_{\{0,1\}}$ & $\text{Shuffle}_{all}$ & $^{{*}{*}{*}}$ & $^{{*}}$ & 0.515 \\
 & $\text{Shuffle}_{interleaved}$ & $\text{Shuffle}_{all}$ & $^{{*}{*}}$ & $^{{*}}$ & 0.134 \\
Sequential & $\text{Shuffle}_{seq}$ & Recover & $^{{*}{*}{*}}$ & 0.467 & 0.562 \\
 & $\text{Shuffle}_{seq}$ & Reinforce & $^{{*}{*}{*}}$ & 0.467 & 0.954 \\
 & Recover & Reinforce & 0.553 & 0.949 & 0.954 \\
\bottomrule
\end{tabular}%
}
\caption{Non-control period counts: pairwise comparisons between individual experiments within each category and architecture (3 pairs per category). Holm correction within each category per architecture; cells show $^

## Optimizer profiles (θ)

Seed **3003** only (matches `REPORT_plots_FINAL.ipynb`):

- **Global:** non-control Cat1 experiments pooled
- **Sequential:** shuffle-seq / recover / reinforce pooled

Mean θ per model (FCN, Inception, ResNet), averaged across optimizers. Pairwise Welch t-tests between models with all θ samples pooled (FCN vs Inception, FCN vs ResNet, Inception vs ResNet), Holm corrected within each category.

In [29]:
from analysis.final.final_experiment_display import rename_experiments_for_labels
from analysis.final.report_optimizer_performance_helper import build_report_theta_long
from analysis.final.report_plots.config import DEFAULT_SEED
from analysis.final.report_plots.experiment_groups import filter_all_results_seed


def rename_experiments(exp_name: str) -> str:
	return rename_experiments_for_labels(exp_name, LABELS)


all_results_seed = filter_all_results_seed(all_results, seed=DEFAULT_SEED)
print(f"seed {DEFAULT_SEED} runs: {len(all_results_seed)} / {len(all_results)}")

df_theta = build_report_theta_long(all_results_seed, rename_experiments)
print(f"theta rows={len(df_theta)}")

theta_summary, theta_pairwise, theta_optimizer_means = print_optimizer_profile_theta_stats(
	df_theta,
	LABELS,
)

print("\n--- θ mean per model (optimizers pooled) ---")
display(theta_summary)

print("\n--- θ per optimizer (Sequential / MNIST) ---")
display(
	theta_optimizer_means[
		(theta_optimizer_means["category"] == "Sequential")
		& (theta_optimizer_means["model_id"] == "dnn_5x64")
	][["optimizer_id", "mean", "n"]]
)

print("\n--- θ model pairwise (all data pooled; Holm within category) ---")
display(
	theta_pairwise[
		[
			"category",
			"comparison",
			"mean_a",
			"mean_b",
			"mean_diff",
			"t_stat",
			"p_value",
			"p_value_holm",
			"significant_005",
			"n_a",
			"n_b",
		]
	]
)

seed 3003 runs: 150 / 150
theta rows=1410
θ profile means per model (all data pooled within category):
Global: FCN=0.0649, Inception=0.9679, ResNet=0.9429 (n=15/15/15)
Sequential: FCN=0.0074, Inception=0.1994, ResNet=0.2862 (n=350/350/350)

θ profile model pairwise (all data pooled; Welch t-test; Holm corrected within each category):
Global:
  FCN vs Inception: Δmean=-0.9030 (0.0649 vs 0.9679), t=-4.715, raw p=3.14e-04, Holm p=9.42e-04, p<0.05=yes (n=15/15)
  FCN vs ResNet: Δmean=-0.8780 (0.0649 vs 0.9429), t=-4.295, raw p=7.12e-04, Holm p=1.42e-03, p<0.05=yes (n=15/15)
  Inception vs ResNet: Δmean=0.0251 (0.9679 vs 0.9429), t=0.090, raw p=9.29e-01, Holm p=9.29e-01, p<0.05=no (n=15/15)
Sequential:
  FCN vs Inception: Δmean=-0.1920 (0.0074 vs 0.1994), t=-11.546, raw p=1.77e-26, Holm p=3.55e-26, p<0.05=yes (n=350/350)
  FCN vs ResNet: Δmean=-0.2788 (0.0074 vs 0.2862), t=-11.971, raw p=5.53e-28, Holm p=1.66e-27, p<0.05=yes (n=350/350)
  Inception vs ResNet: Δmean=-0.0868 (0.1994 vs 0.2862

,category,mean_fcn,mean_inception,mean_resnet,n_fcn,n_inception,n_resnet
0,Global,0.064910,0.967947,0.942871,15,15,15
1,Sequential,0.007368,0.199360,0.286164,350,350,350



--- θ per optimizer (Sequential / MNIST) ---


,optimizer_id,mean,n
15,sgd,0.000183,70
16,adagrad,0.003781,70
17,adam,0.000381,70
18,pure_shampoo,0.000220,70
19,grafted_shampoo,0.032274,70



--- θ model pairwise (all data pooled; Holm within category) ---


,category,comparison,mean_a,mean_b,mean_diff,t_stat,p_value,p_value_holm,significant_005,n_a,n_b
0,Global,FCN vs Inception,0.064910,0.967947,-0.903038,-4.714762,3.141004e-04,9.423011e-04,True,15,15
1,Global,FCN vs ResNet,0.064910,0.942871,-0.877961,-4.294873,7.122116e-04,1.424423e-03,True,15,15
2,Global,Inception vs ResNet,0.967947,0.942871,0.025076,0.089952,9.289681e-01,9.289681e-01,False,15,15
3,Sequential,FCN vs Inception,0.007368,0.199360,-0.191992,-11.545789,1.774685e-26,3.549371e-26,True,350,350
4,Sequential,FCN vs ResNet,0.007368,0.286164,-0.278796,-11.971041,5.525259e-28,1.657578e-27,True,350,350
5,Sequential,Inception vs ResNet,0.199360,0.286164,-0.086804,-3.053641,2.356203e-03,2.356203e-03,True,350,350


## Optimizer profiles (spider)

Seed **3003** only (matches `REPORT_plots_FINAL.ipynb` / `final_optimizer_profiles_spider.pdf`):

- **Global:** non-control Cat1 experiments pooled
- **Sequential:** shuffle-seq / recover / reinforce pooled

`radial_value` is the coordinate drawn on each spider axis (`robustness` uses the multi-scale remap).

Pairwise Welch *t*-tests between optimizers use the same pooled sample-level normalized scores; Holm correction is applied independently within each `(category, model, variable)` group.

In [30]:
from analysis.final.final_compare_all_optimizers_helpers import build_df_long
from analysis.final.report_optimizer_performance_helper import build_report_ttc_long
from analysis.final.report_plots.experiment_groups import cat1_non_control_experiment_labels
from analysis.final.report_plots.inline_stats import (
	compute_optimizer_profile_spider_optimizer_pairwise_holm_wide,
	compute_optimizer_profile_spider_optimizer_pairwise_ttest,
)
from analysis.final.report_plots.report_plots_helper import (
	build_optimizer_profile_spider_pooled_plot_data,
	build_report_spider_means,
	build_report_spider_scores,
)

df_long = build_df_long(all_results_seed, rename_experiments)
df_ttc = build_report_ttc_long(all_results_seed, rename_experiments)
spider_scores = build_report_spider_scores(df_long, df_ttc)
spider_means = build_report_spider_means(df_long, df_ttc)
print(f"spider_scores rows={len(spider_scores)}  spider_means rows={len(spider_means)}")

cat1_pool = cat1_non_control_experiment_labels(
	spider_means["experiment"].unique(), LABELS
)
print(f"Cat1 non-control pool: {cat1_pool}")

spider_plot_data = build_optimizer_profile_spider_pooled_plot_data(
	spider_means,
	labels=LABELS,
	show_angle=False,
)
print(f"spider plot rows={len(spider_plot_data)}")

display(spider_plot_data)

# For the wide table use value_norm for robustness (the plain [0,1] score)
# and radial_value for all other variables (they are identical to value_norm anyway).
_wide_val = spider_plot_data["radial_value"].where(
	spider_plot_data["variable"] != "robustness",
	spider_plot_data["value_norm"],
)
spider_plot_wide = spider_plot_data.assign(_wide_val=_wide_val).pivot_table(
	index=["category", "model", "optimizer_id"],
	columns="variable",
	values="_wide_val",
)
print("\n--- spider radial values (wide) ---")
display(spider_plot_wide)

spider_optimizer_pairwise = compute_optimizer_profile_spider_optimizer_pairwise_ttest(
	spider_scores,
	LABELS,
	show_angle=False,
)
spider_optimizer_pairwise_holm_wide = compute_optimizer_profile_spider_optimizer_pairwise_holm_wide(
	spider_scores,
	LABELS,
	pairwise=spider_optimizer_pairwise,
	show_angle=False,
)
print(
	f"\n--- spider optimizer pairwise Holm p-values "
	f"(per category × model × variable; n={len(spider_optimizer_pairwise_holm_wide)}) ---"
)
display(spider_optimizer_pairwise_holm_wide)

spider_scores rows=47701  spider_means rows=675
Cat1 non-control pool: ['$\\text{Shuffle}_{\\{0,1\\}}$', '$\\text{Shuffle}_{interleaved}$', '$\\text{Shuffle}_{all}$']
spider plot rows=120


,category,model_id,model,optimizer_id,variable,value_norm,radial_value
0,Global,dnn_5x64,FCN,sgd,length,0.058214,0.058214
1,Global,dnn_5x64,FCN,sgd,robustness,0.819847,0.662213
2,Global,dnn_5x64,FCN,sgd,start_onset,0.480528,0.480528
3,Global,dnn_5x64,FCN,sgd,ttc,0.635667,0.635667
4,Global,dnn_5x64,FCN,adagrad,length,0.174524,0.174524
...,...,...,...,...,...,...,...
115,Sequential,resnet32_dnn_5x64,Resnet,pure_shampoo,ttc,0.454222,0.454222
116,Sequential,resnet32_dnn_5x64,Resnet,grafted_shampoo,length,0.501243,0.501243
117,Sequential,resnet32_dnn_5x64,Resnet,grafted_shampoo,robustness,0.805668,0.635628
118,Sequential,resnet32_dnn_5x64,Resnet,grafted_shampoo,start_onset,0.905397,0.905397



--- spider radial values (wide) ---


variable                                length  robustness  start_onset  \
category   model     optimizer_id                                         
Global     FCN       sgd              0.058214    0.819847     0.480528   
                     adagrad          0.174524    0.889357     0.688571   
                     adam             0.319276    0.776245     0.631162   
                     pure_shampoo     0.281903    0.773180     0.590931   
                     grafted_shampoo  0.284344    0.788396     0.637222   
           Inception sgd              0.082306    0.935926     0.559458   
                     adagrad          0.232500    0.918365     0.607222   
                     adam             0.668095    0.853129     0.449048   
                     pure_shampoo     0.655000    0.865092     0.683333   
                     grafted_shampoo  0.731722    0.841373     0.655667   
           Resnet    sgd              0.033148    0.951464     0.437222   
                     adagrad          0.431111    0.866749     0.620000   
                     adam             0.500417    0.907302     0.810000   
                     pure_shampoo     0.790370    0.873426     0.436667   
                     grafted_shampoo  0.607556    0.888841     0.715333   
Sequential FCN       sgd              0.578324    0.702270     0.963599   
                     adagrad          0.612564    0.706588     0.954231   
                     adam             0.538655    0.655164     0.965512   
                     pure_shampoo     0.598541    0.707781     0.963951   
                     grafted_shampoo  0.562329    0.659296     0.955921   
           Inception sgd              0.553274    0.827462     0.949161   
                     adagrad          0.565045    0.815704     0.809251   
                     adam             0.602134    0.769607     0.901880   
                     pure_shampoo     0.516157    0.807967     0.933637   
                     grafted_shampoo  0.609889    0.785307     0.903315   
           Resnet    sgd              0.565759    0.817801     0.952452   
                     adagrad          0.515910    0.765486     0.804545   
                     adam             0.655308    0.749827     0.879034   
                     pure_shampoo     0.532597    0.790766     0.922200   
                     grafted_shampoo  0.501243    0.805668     0.905397   

variable                                   ttc  
category   model     optimizer_id               
Global     FCN       sgd              0.635667  
                     adagrad          0.529667  
                     adam             0.737667  
                     pure_shampoo     0.531333  
                     grafted_shampoo  0.678667  
           Inception sgd              0.818667  
                     adagrad          0.696333  
                     adam             0.815667  
                     pure_shampoo     0.797333  
                     grafted_shampoo  0.772667  
           Resnet    sgd              0.746000  
                     adagrad          0.677000  
                     adam             0.683667  
                     pure_shampoo     0.787333  
                     grafted_shampoo  0.700000  
Sequential FCN       sgd              0.423194  
                     adagrad          0.537876  
                     adam             0.431911  
                     pure_shampoo     0.310667  
                     grafted_shampoo  0.633184  
           Inception sgd              0.652833  
                     adagrad          0.762905  
                     adam             0.763325  
                     pure_shampoo     0.689431  
                     grafted_shampoo  0.717081  
           Resnet    sgd              0.458392  
                     adagrad          0.695875  
                     adam             0.515034  
                     pure_shampoo     0.454222  
                     grafted_shampoo  0.550500


--- spider optimizer pairwise Holm p-values (per category × model × variable; n=24) ---


pair,category,model,variable,p_holm__adagrad__adam,p_holm__adagrad__grafted_shampoo,p_holm__adagrad__pure_shampoo,p_holm__adagrad__sgd,p_holm__adam__grafted_shampoo,p_holm__adam__pure_shampoo,p_holm__adam__sgd,p_holm__grafted_shampoo__pure_shampoo,p_holm__grafted_shampoo__sgd,p_holm__pure_shampoo__sgd
0,Global,FCN,length,1.000000,1.000000,1.000000e+00,1.000000,1.000000,1.000000e+00,0.000743,1.000000e+00,0.003604,0.003604
1,Global,FCN,robustness,1.000000,1.000000,1.000000e+00,1.000000,1.000000,1.000000e+00,1.000000,1.000000e+00,1.000000,1.000000
2,Global,FCN,start_onset,1.000000,1.000000,1.000000e+00,1.000000,1.000000,1.000000e+00,1.000000,1.000000e+00,1.000000,1.000000
3,Global,FCN,ttc,1.000000,1.000000,1.000000e+00,1.000000,1.000000,1.000000e+00,1.000000,1.000000e+00,1.000000,1.000000
4,Global,Inception,length,0.302981,0.266058,2.371695e-01,1.000000,1.000000,1.000000e+00,0.094167,1.000000e+00,0.018861,0.047765
5,Global,Inception,robustness,1.000000,1.000000,1.000000e+00,1.000000,1.000000,1.000000e+00,1.000000,1.000000e+00,0.297317,1.000000
6,Global,Inception,start_onset,1.000000,1.000000,1.000000e+00,1.000000,1.000000,1.000000e+00,1.000000,1.000000e+00,1.000000,1.000000
7,Global,Inception,ttc,1.000000,1.000000,1.000000e+00,1.000000,1.000000,1.000000e+00,1.000000,1.000000e+00,1.000000,1.000000
8,Global,Resnet,length,1.000000,1.000000,1.000000e+00,0.431749,1.000000,1.000000e+00,0.338910,1.000000e+00,0.006545,0.062107
9,Global,Resnet,robustness,1.000000,1.000000,1.000000e+00,0.406726,1.000000,1.000000e+00,1.000000,1.000000e+00,0.175134,0.062507


## High activation angle — selected conditions

Robustness and period-length distributions for BTSP periods with activation angle > 45°:
Control global (✗PT) on MNIST FCN and CIFAR Inception; Reinforce / SGD / FCN.

In [31]:
from collections.abc import Callable

from analysis.dataset_filter import INCEPTION_MODEL_ID, MNIST_MODEL_ID
from analysis.dw_vicinity import robustness_score_column
from analysis.final.report_plots.experiment_groups import classify_report_summary_group
from analysis.final.report_plots.style import report_model_label

ANGLE_DEG_THRESHOLD = 45.0
CONTROL_GLOBAL_NO_PT = "cat1_control_global_nopre"
SGD_OPTIMIZER_ID = "sgd_lr0.01"

HIGH_ANGLE_TARGETS: list[tuple[str, dict]] = [
	(
		f"{LABELS.shuffle_nopre} / {report_model_label(MNIST_MODEL_ID)}",
		{"model_id": MNIST_MODEL_ID, "report_group": CONTROL_GLOBAL_NO_PT},
	),
	(
		f"{LABELS.shuffle_nopre} / {report_model_label(INCEPTION_MODEL_ID)}",
		{"model_id": INCEPTION_MODEL_ID, "report_group": CONTROL_GLOBAL_NO_PT},
	),
	(
		"Reinforce / SGD / FCN",
		{
			"model_id": MNIST_MODEL_ID,
			"experiment_predicate": lambda eid: "cat2_sequence_reinforce" in eid,
			"optimizer_id": SGD_OPTIMIZER_ID,
		},
	),
]


def _collect_high_angle_periods(
	*,
	model_id: str,
	report_group: str | None = None,
	experiment_predicate: Callable[[str], bool] | None = None,
	optimizer_id: str | None = None,
) -> pd.DataFrame:
	rows: list[dict] = []
	for (eid, mid, oid, rid), v in all_results.items():
		if v.get("error") or mid != model_id:
			continue
		if report_group is not None and classify_report_summary_group(eid) != report_group:
			continue
		if experiment_predicate is not None and not experiment_predicate(str(eid)):
			continue
		if optimizer_id is not None and oid != optimizer_id:
			continue
		pdf = v.get("periods_df")
		if pdf is None or pdf.empty:
			continue
		rob_col = robustness_score_column(pdf)
		required = {"activation_angle", "total_length_iter", rob_col}
		if not required.issubset(pdf.columns):
			continue
		sub = pdf.copy()
		sub["activation_angle"] = pd.to_numeric(sub["activation_angle"], errors="coerce")
		sub["total_length_iter"] = pd.to_numeric(sub["total_length_iter"], errors="coerce")
		sub[rob_col] = pd.to_numeric(sub[rob_col], errors="coerce")
		sub = sub.dropna(subset=["activation_angle", "total_length_iter", rob_col])
		sub = sub[sub["activation_angle"] > ANGLE_DEG_THRESHOLD]
		for _, row in sub.iterrows():
			rows.append(
				{
					"experiment_id": eid,
					"optimizer_id": oid,
					"run_id": rid,
					"activation_angle": float(row["activation_angle"]),
					"robustness": float(row[rob_col]),
					"length_iter": float(row["total_length_iter"]),
				}
			)
	return pd.DataFrame(rows)


def _print_high_angle_stats(df: pd.DataFrame, *, header: str) -> None:
	print(f"{header} — periods with activation angle > {ANGLE_DEG_THRESHOLD:.0f}°")
	if df.empty:
		print("(no periods)")
		return
	robustness = df["robustness"]
	length = df["length_iter"]
	rob_q = robustness.quantile([0.25, 0.5, 0.75, 0.9])
	len_q = length.quantile([0.25, 0.5, 0.75, 0.9])
	print(f"  n_periods={len(df)}")
	print(
		f"  robustness: mean={robustness.mean():.4f}  median={rob_q[0.5]:.4f}  "
		f"Q1={rob_q[0.25]:.4f}  Q3={rob_q[0.75]:.4f}  Q90={rob_q[0.9]:.4f}  "
		f"max={robustness.max():.4f}"
	)
	print(
		f"  length (iter): mean={length.mean():.3f}  median={len_q[0.5]:.3f}  "
		f"Q1={len_q[0.25]:.3f}  Q3={len_q[0.75]:.3f}  Q90={len_q[0.9]:.3f}  "
		f"max={length.max():.3f}"
	)


for i, (header, collect_kwargs) in enumerate(HIGH_ANGLE_TARGETS):
	if i:
		print()
	_print_high_angle_stats(_collect_high_angle_periods(**collect_kwargs), header=header)

Control global (✗PT) / FCN — periods with activation angle > 45°
  n_periods=3
  robustness: mean=0.6786  median=0.7127  Q1=0.6413  Q3=0.7330  Q90=0.7452  max=0.7533
  length (iter): mean=3.333  median=3.000  Q1=2.000  Q3=4.500  Q90=5.400  max=6.000

Control global (✗PT) / Inception — periods with activation angle > 45°
  n_periods=78
  robustness: mean=0.9866  median=0.9965  Q1=0.9947  Q3=0.9978  Q90=0.9996  max=0.9999
  length (iter): mean=35.551  median=2.000  Q1=1.000  Q3=8.000  Q90=27.100  max=999.000

Reinforce / SGD / FCN — periods with activation angle > 45°
  n_periods=6
  robustness: mean=0.7066  median=0.7209  Q1=0.6172  Q3=0.8053  Q90=0.8319  max=0.8367
  length (iter): mean=799.833  median=771.500  Q1=424.000  Q3=1260.000  Q90=1400.000  max=1400.000


## Layerwise analysis

*(Placeholder — extend `inline_stats.py` and add cells here.)*